# Week 5 RAG Optimisation - Long-Source Corrective Run

> **Status: LATEST / STANDARD REFERENCE DELIVERABLE (v1.1.0).** This notebook replaces the superseded atomic-section run and is the file reviewers should open for the Week 5 RAG optimisation requirement.

Corrective 3×3×2 full-factorial analysis over 20 frozen Senpai long-document questions. Scores from the local independent Judge are diagnostic, not human ground truth or production-readiness evidence.

In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
if HERE.name != 'phase_c_synthesis':
    HERE = HERE / 'phase_c_synthesis'
analysis = json.loads((HERE / 'W05_RAG_Long_Source_Optimisation_Summary_v1.1.0.json').read_text(encoding='utf-8'))
summary = pd.read_csv(HERE / 'W05_RAG_Long_Source_Optimisation_Summary_v1.1.0.csv')
items = pd.read_csv(HERE / 'W05_RAG_Long_Source_Optimisation_Item_Results_v1.1.0.csv')
len(summary), len(items), analysis['pareto_variant_ids']

(18,
 360,
 ['chunk-1024_topk-3_rerank-ce',
  'chunk-1024_topk-5_rerank-ce',
  'chunk-512_topk-5_rerank-ce'])

In [2]:
summary_view = summary.assign(pareto_optimal=summary['variant_id'].isin(analysis['pareto_variant_ids']))
cols = ['variant_id', 'mean_faithfulness', 'mean_required_point_coverage', 'mean_answer_relevance', 'p50_question_to_response_ms', 'pareto_optimal']
summary_view[cols].sort_values(['pareto_optimal', 'mean_required_point_coverage'], ascending=[False, False])

,variant_id,mean_faithfulness,mean_required_point_coverage,mean_answer_relevance,p50_question_to_response_ms,pareto_optimal
2,chunk-1024_topk-3_rerank-ce,0.836071,0.975000,0.640958,8679.4645,True
4,chunk-1024_topk-5_rerank-ce,0.909524,0.975000,0.663191,10862.9150,True
16,chunk-512_topk-5_rerank-ce,0.901958,0.966667,0.732971,6576.1670,True
8,chunk-256_topk-3_rerank-ce,0.865789,0.950000,0.599607,7617.0830,False
11,chunk-256_topk-5_rerank-none,0.871944,0.950000,0.622855,7325.0490,False
10,chunk-256_topk-5_rerank-ce,0.810606,0.947368,0.713584,7511.8185,False
9,chunk-256_topk-3_rerank-none,0.877500,0.931250,0.713446,7410.5175,False
17,chunk-512_topk-5_rerank-none,0.808333,0.925000,0.607897,6811.5310,False
5,chunk-1024_topk-5_rerank-none,0.846528,0.912500,0.568916,7526.4240,False
14,chunk-512_topk-3_rerank-ce,0.838194,0.912500,0.655842,7678.3360,False


In [3]:
pd.DataFrame(analysis['contrast_factor_summary'])

,factor,contrasts,mean_delta_faithfulness,mean_delta_answer_relevance,mean_delta_required_point_coverage,mean_delta_question_to_response_ms
0,chunk_size_tokens,18,-0.007177,-0.049651,-0.013745,1199.650711
1,reranking,9,0.026297,0.076557,0.066358,772.853667
2,top_k,18,0.062313,0.106294,0.083125,606.713672


In [4]:
pd.DataFrame(analysis['reranking_interaction_by_top_k'])

,fixed_top_k,contrasts,mean_delta_faithfulness,mean_delta_answer_relevance,mean_delta_required_point_coverage,mean_delta_question_to_response_ms
0,1,3,0.063427,0.126168,0.133102,141.729600
1,3,3,-0.016298,0.000144,0.031250,1068.999117
2,5,3,0.031761,0.103359,0.034722,1107.832283


Pareto membership means no tested configuration is at least as good on both diagnostic quality axes and no slower, with a strict improvement on at least one axis. It is conditional on this corpus, question set, model stack, hardware, and run. See `W05_RAG_Long_Source_Optimisation_Report_v1.1.0.md` for the formal limitations.